# CNN/DailyMail Data Preprocessing

This notebook replaces the old XSum preprocessing notebooks.

It performs only the steps needed before splitting:

1. Download a reproducible CNN/DailyMail sample.
2. Clean articles and reference highlights.
3. Create one fixed direct-training prompt per article.
4. Create a structured training target containing:
   - Summary
   - 3–5 Key Points
   - Specialized Vocabulary Explanation
   - Quiz
5. Check whether the complete prompt and complete target fit the OPT-1.3B
   training limit.
6. Save all records with a `fits_training_limit` flag.

Long articles are **not chunked here**. Training/validation use only complete
article–target pairs. Test/deployment chunking is handled inside
`opt-1_3b.ipynb`.

## 0. Setup

In [1]:
import os
import re
import json
import random
from collections import Counter

import pandas as pd

RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
RAW_PATH = os.path.join(RAW_DIR, "cnn_dailymail_raw.jsonl")
OUTPUT_PATH = os.path.join(PROCESSED_DIR, "cnn_processed.jsonl")
TEMPLATES_PATH = os.path.join(PROCESSED_DIR, "prompt_templates.json")

MODEL_NAME = "facebook/opt-1.3b"
SAMPLE_SIZE = 2500
SEED = 42

# Must match the fine-tuning notebook.
MAX_TRAIN_LENGTH = 1536
MAX_TARGET_TOKENS = 320

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
random.seed(SEED)

print(f"Raw output:       {RAW_PATH}")
print(f"Processed output: {OUTPUT_PATH}")

Raw output:       data/raw\cnn_dailymail_raw.jsonl
Processed output: data/processed\cnn_processed.jsonl


## 1. Download CNN/DailyMail

The dataset fields are:

- `article`: full news article
- `highlights`: reference summary/highlights
- `id`: article identifier

Streaming is used so the notebook does not download the full training set.

In [2]:
from datasets import load_dataset

if os.path.exists(RAW_PATH):
    print(f"Raw CNN/DailyMail file already exists at '{RAW_PATH}', skipping download.")
else:
    print("Streaming CNN/DailyMail 3.0.0 from Hugging Face...")
    dataset = load_dataset(
        "abisee/cnn_dailymail",
        "3.0.0",
        split="train",
        streaming=True,
    )

    dataset = dataset.shuffle(seed=SEED, buffer_size=10000)

    count = 0
    with open(RAW_PATH, "w", encoding="utf-8") as f:
        for row in dataset:
            record = {
                "article_id": row.get("id"),
                "article": row.get("article"),
                "summary": row.get("highlights"),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            count += 1
            if count >= SAMPLE_SIZE:
                break

    print(f"Saved {count} raw CNN/DailyMail records to '{RAW_PATH}'")

Streaming CNN/DailyMail 3.0.0 from Hugging Face...
Saved 2500 raw CNN/DailyMail records to 'data/raw\cnn_dailymail_raw.jsonl'


## 2. Load, clean and remove duplicates

In [3]:
records = []
with open(RAW_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

raw_df = pd.DataFrame(records)
print(f"Loaded {len(raw_df)} raw records")


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"[\x00-\x1f\x7f]", " ", text)
    text = re.sub(r"[-*_=~]{3,}", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


clean_df = raw_df.dropna(subset=["article", "summary"]).copy()
clean_df["article"] = clean_df["article"].apply(clean_text)
clean_df["summary"] = clean_df["summary"].apply(clean_text)
clean_df = clean_df[
    clean_df["article"].astype(bool) &
    clean_df["summary"].astype(bool)
]
clean_df = clean_df.drop_duplicates(subset=["article"], keep="first")
clean_df = clean_df.reset_index(drop=True)

print(f"{len(clean_df)} records remain after cleaning")
clean_df.head()

Loaded 2500 raw records
2500 records remain after cleaning


,article_id,article,summary
0,9447bac3741cfc5c0ad64ba2716c6dc28018ae40,Celtic completed a deadline-busting swoop for ...,John Guidetti travelled to Glasgow for talks o...
1,8cd6e3574fc2e90608d5d977f2247675e23160b7,British taxpayers are donating thousands of po...,The Mtaani Inititative is part of a £6.7millio...
2,acbf542dc97fe262cabc8f82f852f2d14e80ac60,Manchester City have told UEFA chiefs that wea...,Premier League champions have met with UEFA pr...
3,1faf7efdcec81e90a5aa35fff2963e3bae7f605f,A picture has emerged of gun victim Reeva Stee...,Model looks every inch the accomplished shoote...
4,1cefec398cea732d452a1980dcc0744d9d6ee918,A 101-year-old woman has been evicted from her...,Texana Hollis's son failed to pay her property...


## 3. Build structured targets

CNN/DailyMail only supplies `article` and `highlights`. The extra sections are
created as simple, auditable synthetic labels:

- **Key Points:** primarily split from the reference highlights.
- **Vocabulary:** terms must occur in the article; definitions come from WordNet.
- **Quiz:** questions are created from key points, and each answer is one of the
  reference-supported key points.

These labels are useful for teaching the output format, but they are not
official CNN/DailyMail ground truth.

In [4]:
import nltk
from nltk.corpus import wordnet as wn

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")
WORD_RE = re.compile(r"[A-Za-z][A-Za-z'-]+")

STOPWORDS = {
    "about", "after", "again", "against", "almost", "among", "because",
    "before", "being", "between", "could", "during", "every", "first",
    "from", "have", "into", "more", "most", "other", "over", "same",
    "should", "some", "such", "than", "that", "their", "there", "these",
    "they", "this", "those", "through", "under", "very", "what", "when",
    "where", "which", "while", "with", "would", "your"
}


def split_sentences(text: str):
    return [
        sentence.strip(" -•")
        for sentence in SENTENCE_SPLIT_RE.split(text.strip())
        if sentence.strip(" -•")
    ]


def normalize_point(text: str) -> str:
    text = re.sub(r"^[•\-*\d.)\s]+", "", text).strip()
    if text and text[-1] not in ".!?":
        text += "."
    return text


def build_key_points(summary: str, article: str, minimum=3, maximum=5):
    points = []
    seen = set()

    # CNN highlights are the strongest source for key points.
    for sentence in split_sentences(summary):
        point = normalize_point(sentence)
        key = point.lower()
        if point and key not in seen:
            points.append(point)
            seen.add(key)
        if len(points) >= maximum:
            break

    # Rare fallback when highlights contain fewer than three sentences.
    if len(points) < minimum:
        for sentence in split_sentences(article):
            point = normalize_point(sentence)
            key = point.lower()
            if len(point.split()) >= 6 and key not in seen:
                points.append(point)
                seen.add(key)
            if len(points) >= minimum:
                break

    return points[:maximum]


def wordnet_definition(word: str):
    synsets = wn.synsets(word.lower())
    if not synsets:
        return None
    definition = synsets[0].definition().strip()
    return definition[0].upper() + definition[1:] if definition else None


def build_vocabulary(article: str, maximum=3):
    words = WORD_RE.findall(article)
    frequencies = Counter(word.lower() for word in words)

    candidates = []
    seen = set()

    # Prefer longer, less frequent words that appear in the article.
    for word in sorted(
        frequencies,
        key=lambda w: (-len(w), frequencies[w], w),
    ):
        if (
            len(word) < 7 or
            word in STOPWORDS or
            word in seen or
            word.endswith("'s")
        ):
            continue

        definition = wordnet_definition(word)
        if not definition:
            continue

        candidates.append({
            "term": word,
            "definition": definition,
        })
        seen.add(word)

        if len(candidates) >= maximum:
            break

    return candidates


def extract_subject(point: str):
    # Simple leading proper-name phrase, for example "Manuel Pellegrini".
    match = re.match(
        r"^([A-Z][A-Za-z'-]*(?:\s+[A-Z][A-Za-z'-]*){0,3})\b",
        point,
    )
    return match.group(1) if match else None


def build_quiz(key_points, maximum=3):
    quiz = []

    for index, point in enumerate(key_points[:maximum], start=1):
        subject = extract_subject(point)

        if re.search(r"\d", point):
            question = "What numerical detail is reported in the article?"
        elif subject:
            question = f"What does the article report about {subject}?"
        elif index == 1:
            question = "What is the main event reported in the article?"
        else:
            question = f"What is another important fact reported in the article?"

        quiz.append({
            "question": question,
            "answer": point,
        })

    return quiz


def build_structured_target(row):
    key_points = build_key_points(row["summary"], row["article"])
    vocabulary = build_vocabulary(row["article"])
    quiz = build_quiz(key_points)

    key_points_text = "\n".join(
        f"- {point}" for point in key_points
    )

    vocabulary_text = "\n".join(
        f"- {item['term']}: {item['definition']}"
        for item in vocabulary
    )

    quiz_text = "\n".join(
        f"{index}. {item['question']}\n"
        f"   Answer: {item['answer']}"
        for index, item in enumerate(quiz, start=1)
    )

    return (
        f"Summary:\n{row['summary']}\n\n"
        f"Key Points:\n{key_points_text}\n\n"
        f"Specialized Vocabulary Explanation:\n{vocabulary_text}\n\n"
        f"Quiz:\n{quiz_text}"
    )


structured_df = clean_df.copy()
structured_df["key_points"] = structured_df.apply(
    lambda row: build_key_points(row["summary"], row["article"]),
    axis=1,
)
structured_df["vocabulary"] = structured_df["article"].apply(build_vocabulary)
structured_df["quiz"] = structured_df["key_points"].apply(build_quiz)
structured_df["training_target"] = structured_df.apply(
    build_structured_target,
    axis=1,
)

structured_df[
    ["summary", "key_points", "vocabulary", "quiz", "training_target"]
].head(1)

,summary,key_points,vocabulary,quiz,training_target
0,John Guidetti travelled to Glasgow for talks o...,[John Guidetti travelled to Glasgow for talks ...,"[{'term': 'disappointment', 'definition': 'A f...",[{'question': 'What does the article report ab...,Summary:\nJohn Guidetti travelled to Glasgow f...


## 4. Build the direct training prompt

In [5]:
TRAIN_PROMPT_TEMPLATE = """You are a news simplification assistant. Read the following news article and produce exactly four labelled sections using clear and simple English:

Summary:
Key Points:
Specialized Vocabulary Explanation:
Quiz:

Rules:
- Use only information supported by the article.
- Include 3-5 key points.
- Explain 3 specialized words that appear in the article.
- Include 2-3 factual quiz questions with answers.

Article:
{article}

Response:
"""

CHUNK_PROMPT_TEMPLATE = """Read this part of a news article and write a short factual summary of only this part. Preserve important names, dates, locations and numbers. Do not add outside information.

Article part:
{chunk}

Response:
Summary:
"""

FINAL_PROMPT_TEMPLATE = """You are a news simplification assistant. The text below contains short summaries of consecutive parts of one article. Combine them and produce exactly four labelled sections:

Summary:
Key Points:
Specialized Vocabulary Explanation:
Quiz:

Rules:
- Use only information supported by the part summaries.
- Include 3-5 key points.
- Explain 3 specialized words that appear in the part summaries.
- Include 2-3 factual quiz questions with answers.

Part summaries:
{chunk_summaries}

Response:
"""

prompt_df = structured_df.copy()
prompt_df["prompt"] = prompt_df["article"].apply(
    lambda article: TRAIN_PROMPT_TEMPLATE.format(article=article)
)

print(prompt_df.loc[0, "prompt"][:1000])

You are a news simplification assistant. Read the following news article and produce exactly four labelled sections using clear and simple English:

Summary:
Key Points:
Specialized Vocabulary Explanation:
Quiz:

Rules:
- Use only information supported by the article.
- Include 3-5 key points.
- Explain 3 specialized words that appear in the article.
- Include 2-3 factual quiz questions with answers.

Article:
Celtic completed a deadline-busting swoop for Stefan Scepovic but missed out on a second striker when a deal for Manchester City's John Guidetti collapsed just after midnight due to the paperwork failing to go through in time. Serbian international Scepovic became the only transfer fee paid in the Scottish window when his second u-turn in 24 hours teed up a £2.2million move from Sporting Gijon. And, hours after shipping out Amido Balde, Teemu Pukki, Holmbert Fridjonsson on loan, Celtic manager Ronny Deila thought he’d made it a double with a 12-month deal to take Guidetti from Ma

## 5. Check complete training-pair length

A row is marked `fits_training_limit=True` only when:

- the full target is at most `MAX_TARGET_TOKENS`; and
- the complete prompt plus complete target is at most `MAX_TRAIN_LENGTH`.

Nothing is silently truncated here.

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def count_tokens(text: str, add_special_tokens: bool) -> int:
    return len(
        tokenizer(
            text,
            add_special_tokens=add_special_tokens,
            truncation=False,
        )["input_ids"]
    )


length_df = prompt_df.copy()
length_df["prompt_token_length"] = length_df["prompt"].apply(
    lambda text: count_tokens(text, add_special_tokens=True)
)
length_df["target_token_length"] = length_df["training_target"].apply(
    lambda text: count_tokens(
        text + tokenizer.eos_token,
        add_special_tokens=False,
    )
)
length_df["total_training_tokens"] = (
    length_df["prompt_token_length"] +
    length_df["target_token_length"]
)
length_df["fits_training_limit"] = (
    (length_df["target_token_length"] <= MAX_TARGET_TOKENS) &
    (length_df["total_training_tokens"] <= MAX_TRAIN_LENGTH)
)

fit_count = int(length_df["fits_training_limit"].sum())
print(
    f"Complete training pairs that fit: {fit_count}/{len(length_df)} "
    f"({fit_count / len(length_df):.2%})"
)
print(
    "Long/non-fitting records retained for possible test-time chunking:",
    int((~length_df["fits_training_limit"]).sum()),
)

length_df[
    [
        "prompt_token_length",
        "target_token_length",
        "total_training_tokens",
        "fits_training_limit",
    ]
].describe()

Complete training pairs that fit: 1364/2500 (54.56%)
Long/non-fitting records retained for possible test-time chunking: 1136


,prompt_token_length,target_token_length,total_training_tokens
count,2500.000000,2500.000000,2500.000000
mean,926.901600,310.211200,1237.112800
std,439.146096,55.508809,459.653289
min,198.000000,199.000000,404.000000
25%,603.000000,269.000000,897.750000
50%,827.000000,302.000000,1132.000000
75%,1145.250000,341.000000,1474.000000
max,2469.000000,634.000000,2912.000000


## 6. Save processed records and prompt templates

In [7]:
final_columns = [
    "article_id",
    "article",
    "summary",
    "prompt",
    "training_target",
    "key_points",
    "vocabulary",
    "quiz",
    "prompt_token_length",
    "target_token_length",
    "total_training_tokens",
    "fits_training_limit",
]

final_df = length_df[final_columns].reset_index(drop=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for record in final_df.to_dict(orient="records"):
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

with open(TEMPLATES_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "train_prompt_template": TRAIN_PROMPT_TEMPLATE,
            "single_prompt_template": TRAIN_PROMPT_TEMPLATE,
            "chunk_prompt_template": CHUNK_PROMPT_TEMPLATE,
            "final_prompt_template": FINAL_PROMPT_TEMPLATE,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Saved {len(final_df)} records to '{OUTPUT_PATH}'")
print(f"Saved prompt templates to '{TEMPLATES_PATH}'")

Saved 2500 records to 'data/processed\cnn_processed.jsonl'
Saved prompt templates to 'data/processed\prompt_templates.json'
